# Follow-up experiments: cell B (Colab, T4)

Runs the experiments of **cell B** (CIFAR-10 × BN CNN), because the benchmark runs of cell B were made on Colab:
**E1** (2×2 design) → **E3** (weight-norm recording) → **E2** (weight-decay sweep) → **X1** (LR search with shortened schedules).
About 8.5 T4 hours. Cells D and E run in parallel on Kaggle (`kaggle_exp.ipynb`). Later additions (E2b, E4) use the same notebook with `EXPS` changed.

- Code: upload a zip of this repository as `adam-bench-v3.zip` to `MyDrive` (the notebook unpacks it), or upload the folder to `MyDrive/adam-bench-v3`.
- Results are written straight to `MyDrive/adam-bench-v3/results_exp`, so a lost session loses no data.
- Runtime → Change runtime type → **T4 GPU**.
- Interrupted session: reopen, **Runtime → Run all**. Finished runs are skipped; unfinished runs resume from their checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/adam-bench-v3'   # change if stored elsewhere
RESULTS = f'{PROJECT}/results_exp'
WORK = '/content/adam-bench-v3'
DATA = '/content/data'
CELLS = ['B']
EXPS = ['E1', 'E3', 'E2', 'X1']      # priority order
RUN_SELFTEST = True                  # True the first time; set False after SELFTEST PASSED

import os, zipfile
ZIP = '/content/drive/MyDrive/adam-bench-v3.zip'      # option 2: just upload the zip to MyDrive
if os.path.exists(ZIP):   # always unpack the latest version (the zip holds no results, so nothing is overwritten)
    zipfile.ZipFile(ZIP).extractall('/content/drive/MyDrive')
    print('Unpacked', ZIP)
assert os.path.exists(f'{PROJECT}/run_exp.py'), f'Code not found at {PROJECT}'
!rsync -a --delete --exclude results_exp --exclude data {PROJECT}/ {WORK}/
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# === Pin the torch + cuDNN versions of the benchmark (the runner is a subprocess, so NO restart is needed) ===
import subprocess, sys
WANT = {'B': ('2.11.0', '0.26.0', 'cu130', 92700),
        'D': ('2.11.0', '0.26.0', 'cu128', 91900),
        'E': ('2.11.0', '0.26.0', 'cu128', 91900)}
tv, vv, cu, cudnn_want = WANT[CELLS[0]]
want = f'{tv}+{cu}'

def probe():
    out = subprocess.run([sys.executable, '-c',
                          'import torch; print(torch.__version__); print(torch.backends.cudnn.version()); '
                          'print("|".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))'],
                         capture_output=True, text=True).stdout.splitlines()
    return out[0], int(out[1]), out[2] if len(out) > 2 else ''

have, cudnn, gpus = probe()
print('installed torch:', have, '| required:', want)
if have != want:
    print('Reinstalling torch to match the benchmark (a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'torch=={tv}+{cu}', f'torchvision=={vv}+{cu}',
                           '--index-url', f'https://download.pytorch.org/whl/{cu}'])
    have, cudnn, gpus = probe()
    print('torch after install:', have)
assert have == want, f'Could not install torch {want}; stopping.'

if cudnn != cudnn_want:
    pkg = 'nvidia-cudnn-cu13' if cu.startswith('cu13') else 'nvidia-cudnn-cu12'
    spec = f'{pkg}=={cudnn_want // 10000}.{cudnn_want // 100 % 100}.*'
    print(f'cuDNN is {cudnn}, required {cudnn_want} -> installing {spec}')
    subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', spec])
    have, cudnn, gpus = probe()
print('torch', have, '| cuDNN', cudnn, '(required', cudnn_want, ') | GPU:', gpus)
assert 'T4' in gpus, 'Not a T4 GPU; switch the runtime/accelerator to T4.'
FORCE_ENV = cudnn != cudnn_want
if FORCE_ENV:
    print('WARNING: could not match the cuDNN version. Continuing with --force-env: torch and GPU match, '
          'only the cuDNN patch differs; results are statistically comparable but may not match digit for digit.')
else:
    print('Environment matches the benchmark.')


In [ ]:
# === Self-test (first time, ~3 minutes): must end with SELFTEST PASSED ===
if RUN_SELFTEST:
    !python run_exp.py --selftest --out /content/tmp_selftest
    !python run_cell.py --selftest --out /content/tmp_selftest

In [ ]:
# === Run ===
import subprocess, sys
!python run_exp.py --plan --cells B
p = subprocess.Popen([sys.executable, '-u', 'run_exp.py', '--exp', *EXPS, '--cells', *CELLS,
                      '--out', RESULTS, '--data', DATA] + (['--force-env'] if FORCE_ENV else []),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print('Exit code', rc, '(2 = environment/configuration error, see the messages above)')

In [ ]:
# === Progress ===
!python run_exp.py --status --cells B --out {RESULTS}